In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

In [2]:
df=pd.read_csv('diabetes.csv')

In [3]:
column=['Glucose','BloodPressure','BMI']
df[column]=df[column].replace(0,np.nan)

In [4]:
x_train,x_test,y_train,y_test=train_test_split(df.drop(columns=['Outcome']),df['Outcome'],random_state=42,test_size=0.2)

In [5]:
x_train.sample()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
645,2,157.0,74.0,35,440,39.4,0.134,30


In [16]:
ct1=ColumnTransformer([
    ('Impute_glucose',SimpleImputer(strategy='median'),[1]),
    ('Impute_BMI',SimpleImputer(strategy='median'),[5]),
    ('Impute_bloodpressure',SimpleImputer(),[2]),
],remainder='passthrough')

In [17]:
ct2=ColumnTransformer([
    ('scaling',RobustScaler(),[0,1,2,3,4,5,6,7])
],remainder='passthrough')

In [19]:
pipe=Pipeline([
    ('ct1',ct1),
    ('ct2',ct2),
])

In [22]:
x_train_transform=pipe.fit_transform(x_train)

In [27]:
x_test_transform=pipe.fit_transform(x_test)

### Logistics Regression model

In [14]:
def gd(X,y):
    
    X = np.insert(X,0,1,axis=1)
    weights = np.ones(X.shape[1])
    lr = 0.5
    
    for i in range(5000):
        y_hat = sigmoid(np.dot(X,weights))
        weights = weights + lr*(np.dot((y-y_hat),X)/X.shape[0])
        
    return weights[1:],weights[0]
        

In [15]:
def sigmoid(z):
    return 1/(1 + np.exp(-z))

In [51]:
def predict(x):
    y= np.where(
            x>0,
            1,
            0
        )
    return y

#### Training model

In [29]:
w,w0=gd(x_train_transform,y_train)

#### Testing model

In [46]:
y_output=np.dot(x_test_transform,w)+w0

In [52]:
y_pred=predict(y_output)

In [53]:
y_pred

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0,
       0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1,
       0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1,
       0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0])

In [54]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)

In [55]:
accuracy

0.7792207792207793

In [44]:
from sklearn.metrics import confusion_matrix, classification_report


print(confusion_matrix(y_test, y_pred))


print(classification_report(y_test, y_pred))

[[86 13]
 [21 34]]
              precision    recall  f1-score   support

           0       0.80      0.87      0.83        99
           1       0.72      0.62      0.67        55

    accuracy                           0.78       154
   macro avg       0.76      0.74      0.75       154
weighted avg       0.78      0.78      0.77       154

